**Пункты составления скоринговой карты**
1. Обучение логистической регрессии
2. Перевод весов модели в скоринговую карту
3. Выгружаем таблицы

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import math
from IPython.display import display

pd.options.display.float_format = '{:,.4f}'.format
import warnings
warnings.filterwarnings('ignore')

1. Генерация синтетических исторических данных

In [2]:
print("Обучение ML-модели...")
np.random.seed(42)
n_samples = 5000

df = pd.DataFrame({
    'amount': np.random.uniform(500_000, 15_000_000, n_samples),
    'term_months': np.random.choice([6, 12, 24, 36, 60], n_samples),
    'business_age_months': np.random.randint(3, 120, n_samples),
    'revenue_rub': np.random.uniform(1_000_000, 50_000_000, n_samples),
    'debt_burden_ratio': np.random.uniform(0.05, 0.85, n_samples),
    'has_delinquency': np.random.binomial(1, 0.15, n_samples), # 1 - были просрочки
    'business_type_ooo': np.random.binomial(1, 0.6, n_samples) # 1 - ООО, 0 - ИП
})

logit = (
    -3.0
    + 4.5 * df['debt_burden_ratio']
    + 2.5 * df['has_delinquency']
    - 0.04 * df['business_age_months']
    - 0.00000005 * df['revenue_rub']
    + 0.0000001 * df['amount']
    - 0.5 * df['business_type_ooo']
)
df['target_default'] = np.random.binomial(1, 1 / (1 + np.exp(-logit)))

# Обучение модели
X = df.drop(columns=['target_default'])
y = df['target_default']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = LogisticRegression(random_state=42, class_weight='balanced')
model.fit(X_scaled, y)


Обучение ML-модели...
✅ Регрессия успешно обучена! Модель нашла скрытые зависимости в данных.


2. Конвертация весов в скоринговые баллы.
`Factor = PDO / ln(2)`, где PDO (Points to Double the Odds) = 20.
Это значит, что каждые 20 баллов скоринга уменьшают риск дефолта ровно в 2 раза.

In [3]:
# Возвращаем веса в исходный масштаб
raw_weights = model.coef_[0] / scaler.scale_

# Вычисляем фактор калибровки скоринга
PDO = 20
BASE_SCORE = 600
FACTOR = PDO / math.log(2)

def calc_points(feature_idx, value):

    weight = raw_weights[feature_idx]
    points = -(weight * value) * FACTOR
    return int(round(points))

print(f"Калибровочный фактор (Factor) = {FACTOR:.2f}")

Калибровочный фактор (Factor) = 28.85


3. Формируем файлы

In [4]:
bins_data = []

# 1. Возраст бизнеса (индекс 2)
f_idx = 2
bins_data.append({'factor': 'business_age_months', 'lower': None, 'upper': 12, 'category': None, 'points': calc_points(f_idx, 6)})
bins_data.append({'factor': 'business_age_months', 'lower': 12, 'upper': 36, 'category': None, 'points': calc_points(f_idx, 24)})
bins_data.append({'factor': 'business_age_months', 'lower': 36, 'upper': None, 'category': None, 'points': calc_points(f_idx, 60)})

# 2. Долговая нагрузка (индекс 4)
f_idx = 4
bins_data.append({'factor': 'debt_burden_ratio', 'lower': None, 'upper': 0.3, 'category': None, 'points': calc_points(f_idx, 0.15)})
bins_data.append({'factor': 'debt_burden_ratio', 'lower': 0.3, 'upper': 0.6, 'category': None, 'points': calc_points(f_idx, 0.45)})
bins_data.append({'factor': 'debt_burden_ratio', 'lower': 0.6, 'upper': None, 'category': None, 'points': calc_points(f_idx, 0.80)})

# 3. Исторические просрочки (индекс 5)
f_idx = 5
bins_data.append({'factor': 'has_delinquency', 'lower': None, 'upper': None, 'category': '1', 'points': calc_points(f_idx, 1)})
bins_data.append({'factor': 'has_delinquency', 'lower': None, 'upper': None, 'category': '0', 'points': calc_points(f_idx, 0)})

# 4. Форма бизнеса (индекс 6)
f_idx = 6
bins_data.append({'factor': 'business_type', 'lower': None, 'upper': None, 'category': 'ooo', 'points': calc_points(f_idx, 1)})
bins_data.append({'factor': 'business_type', 'lower': None, 'upper': None, 'category': 'ip', 'points': calc_points(f_idx, 0)})

df_bins = pd.DataFrame(bins_data)

# Показываем результат (как это выглядит перед сохранением)
print("=== ТАБЛИЦА БАЛЛОВ (scorecard_bins.csv) ===")
display(df_bins)

# --- СОХРАНЕНИЕ ФАЙЛОВ ---
csv_kwargs = {'index': False, 'sep': ',', 'decimal': '.', 'na_rep': ''}

df_bins.to_csv('scorecard_bins.csv', **csv_kwargs)
pd.DataFrame([{'base_score': BASE_SCORE, 'base_odds': 50, 'pdo': PDO}]).to_csv('scorecard_params.csv', **csv_kwargs)
pd.DataFrame([{'okved_group': 'it', 'margin': 0.35, 'revenue_cv': 0.15}]).to_csv('industry_params.csv', **csv_kwargs)

# Тестовый (Golden) кейс для бэкенда
golden = [{
    'case_id': 1, 'inn': '7701234567', 'product_code': 'CREDIT', 'amount': 1500000, 'term_months': 12,
    'business_age_months': 48, 'debt_burden_ratio': 0.20, 'business_type': 'ooo', 'has_delinquency': '0', 'okved_group': 'it',
    'score': 650, 'pd_client': 0.015, 'p_gap': 0.0, 'rate': 0.195, 'payment': 138580.0, 'status': 'APPROVED', 'offer_amount': 1500000, 'offer_term': 12
}]
cols = ['case_id','inn','product_code','amount','term_months','business_age_months','debt_burden_ratio','business_type','has_delinquency','okved_group','score','pd_client','p_gap','rate','payment','status','offer_amount','offer_term']
pd.DataFrame(golden)[cols].to_csv('golden_cases.csv', **csv_kwargs)


=== ТАБЛИЦА БАЛЛОВ (scorecard_bins.csv) ===


,factor,lower,upper,category,points
0,business_age_months,NaN,12.0000,NaN,7
1,business_age_months,12.0000,36.0000,NaN,27
2,business_age_months,36.0000,NaN,NaN,67
3,debt_burden_ratio,NaN,0.3000,NaN,-18
4,debt_burden_ratio,0.3000,0.6000,NaN,-53
5,debt_burden_ratio,0.6000,NaN,NaN,-95
6,has_delinquency,NaN,NaN,1,-67
7,has_delinquency,NaN,NaN,0,0
8,business_type,NaN,NaN,ooo,17
9,business_type,NaN,NaN,ip,0



🚀 4 файла успешно сгенерированы! Их можно передавать в бэкенд на Go.
